In [19]:
# arcgis_search_tool.py
import arcpy
from arcgis.gis import GIS
import os
import json
from typing import List, Dict, Any, Optional, Union
from datetime import datetime
from dotenv import load_dotenv
from langchain.tools import tool

# Load environment variables from .env file
load_dotenv()

True

In [20]:
# --- Authentication Setup ---

ARCGIS_URL = "https://www.arcgis.com"
ARC_USER = os.environ.get("ARC_USER")
ARC_PASS = os.environ.get("ARC_PASS")

In [21]:
def get_gis_connection() -> Optional[GIS]:
    """
    Helper function to establish GIS connection.
    Handles authentication securely using username/password from environment variables.
    Returns an anonymous connection if authentication fails or credentials are not set.
    """
    if ARC_USER and ARC_PASS:
        try:
            print(f"Attempting to connect to {ARCGIS_URL} as user {ARC_USER}...")
            gis = GIS(ARCGIS_URL, username=ARC_USER, password=ARC_PASS)
            # Optionally, check if the connection is valid by accessing user profile
            print(f"Successfully connected to {gis.url} as user {gis.properties.user.username}")
            return gis
        except Exception as e:
            print(f"Error connecting to ArcGIS with username/password: {e}")
            print("Falling back to anonymous connection. Functionality may be limited.")
            gis = GIS(ARCGIS_URL)
            return gis
    else:
        print("Warning: ARC_USER or ARC_PASS environment variable not set.")
        print("Attempting anonymous connection. Functionality may be limited.")
        try:
            gis = GIS(ARCGIS_URL)
            print(f"Successfully established anonymous connection to {gis.url}")
            return gis
        except Exception as e:
            print(f"Error establishing anonymous connection to ArcGIS: {e}")
            return None

In [ ]:
def search_arcgis_content(
    query: str,
    title: Optional[str] = None,
    item_type: Optional[str] = None,
    owner: Optional[str] = None,
    tags: Optional[Union[str, List[str]]] = None,
    typeKeywords: Optional[Union[str, List[str]]] = None,
    snippet: Optional[str] = None,
    group_id: Optional[str] = None,
    categories: Optional[Union[str, List[str]]] = None,
    created_start_date: Optional[str] = None, # YYYY-MM-DD
    created_end_date: Optional[str] = None,   # YYYY-MM-DD
    max_results: int = 10,
    search_living_atlas_focused: bool = False,
    search_outside_org: bool = True # Renamed from search_public
) -> str:
    """Searches ArcGIS Online/Portal for GIS items using advanced filtering based on the REST API query syntax.

    Allows flexible searching using keywords and various filters like title, item type, owner,
    tags, typeKeywords, description, snippet, created date, group ID,
    and categories. Constructs queries following ArcGIS REST API standards.
    Does NOT perform complex spatial filtering (like polygon intersects) during the item search itself;
    use location names in the 'query' argument for geographic focus.
    Results are sorted by relevance descending by default.

    Suggestions for improving search results:
    - Use specific keywords or phrases in the query (e.g., 'california population density').
    - Searching using the title and snippet fields often yields the best results if described properly.
    
    GIS Concepts:
    - Content Search: Finding GIS items based on metadata. Uses Lucene query syntax.
    - Item Metadata Fields: title, tags, snippet, description, type, typeKeywords, owner, created, id, access, categories, group.
    - Living Atlas Focus: Option to prioritize searching common Living Atlas sources if no specific owner/group/tags are given.

    Args:
        query: The primary keyword search string (e.g., "california population density", "hospitals near main street london"). Can include boolean operators (AND, OR, NOT) and field searches (e.g., 'title:"San Francisco"').
        title: (Optional) Filter by item title. Exact match using 'title:"value"'.
        item_type: (Optional) Filter by item type (e.g., 'Feature Service', 'Web Map'). Use exact case and quotes: 'type:"Web Map"'.
        owner: (Optional) Filter by the username of the item owner (e.g., 'esri', 'fedmaps_usgs'). Uses 'owner:"value"'.
        tags: (Optional) Filter by tags. Single tag string or list. Items must have ALL specified tags. Uses 'tags:"value"' or '(tags:"tag1" AND tags:"tag2")'.
        typeKeywords: (Optional) Filter by type keywords. Single string or list. Uses 'typeKeywords:"value"'.
        snippet: (Optional) Search within the item snippet (summary). Uses 'snippet:"value"'.
        group_id: (Optional) Filter by the ID of a specific group. Uses 'group:"value"'.
        categories: (Optional) Filter by organization content categories. Single string or list. Uses 'categories:"value"'.
        created_start_date: (Optional) Filter items created on or after this date (YYYY-MM-DD).
        created_end_date: (Optional) Filter items created on or before this date (YYYY-MM-DD).
        max_results: (Optional) Max number of results (default: 10).
        search_living_atlas_focused: (Optional) If True and owner/tags/group_id are NOT specified, adds filters for common Living Atlas owners/tags.
        search_outside_org: (Optional) If True (default), searches content outside the user's organization (implies public or shared content depending on context). If False, searches only within the user's organization. This controls the `outside_org` parameter of `gis.content.search`. An `access:public` filter is added explicitly if True.

    Returns:
        A JSON string representing a list of found items (including title, id, type, owner, snippet, description, tags, created date).
        Returns an error message string on failure or if no items are found.

    Example Use by AI:
        # User: "Find recent wildfire Web Maps in California from CALFIRE_Agency"
        >>> search_arcgis_content(query="wildfire california", owner="CALFIRE_Agency", created_start_date="2024-01-01", item_type="Web Map") # Example uses created_start_date now

        # User: "Show me authoritative elevation layers for Mount Rainier area"
        >>> search_arcgis_content(query="elevation DEM Mount Rainier", item_type="Imagery Layer", search_living_atlas_focused=True) # Removed contentStatus

        # User: "Search for census data tagged 'population' and '2020' with type keyword 'Demographics'"
        >>> search_arcgis_content(query="census", tags=["population", "2020"], typeKeywords="Demographics", item_type="Feature Service")

        # User: "Find public transport layers for London created this year within org 'MyOrgID'"
        >>> search_arcgis_content(query="public transport london", item_type="Feature Service", created_start_date="2025-01-01", search_outside_org=False) # Removed orgid, Assuming current year 2025
    """
    gis = get_gis_connection()
    if not gis:
        return "Error: Failed to establish connection to ArcGIS. Check Credentials/Network."

    # --- Input Validation ---
    if not isinstance(query, str): # Allow empty query if other filters are used
        return "Error: query must be a string."
    if not isinstance(max_results, int) or max_results <= 0:
        return "Error: max_results must be a positive integer."

    def _validate_and_format_date_for_api(date_str: Optional[str]) -> Optional[int]:
        """Parses YYYY-MM-DD input and returns milliseconds since epoch for API query."""
        if not date_str: return None
        try:
            # Parse the date string
            dt = datetime.strptime(date_str, "%Y-%m-%d")
            # Convert to timestamp (seconds since epoch) and then to milliseconds
            timestamp_ms = int(dt.timestamp() * 1000)
            return timestamp_ms
        except ValueError:
            raise ValueError(f"Invalid date format '{date_str}'. Use YYYY-MM-DD.")

    def _format_list_filter(field_name: str, values: Optional[Union[str, List[str]]]) -> Optional[str]:
        """Formats a list of values for a field query (e.g., tags, typeKeywords)."""
        if not values: return None
        value_list = [values] if isinstance(values, str) else values
        processed_values = []
        for val in value_list:
            if isinstance(val, str) and val.strip():
                # Quote values, especially if they contain spaces
                val_formatted = f'"{val.strip()}"'
                processed_values.append(f'{field_name}:{val_formatted}')
        if not processed_values: return None
        # Combine multiple values with AND logic as per documentation examples
        return "(" + " AND ".join(processed_values) + ")" if len(processed_values) > 1 else processed_values[0]

    try:
        # Validate dates before building query
        created_start_ms = _validate_and_format_date_for_api(created_start_date)
        created_end_ms = _validate_and_format_date_for_api(created_end_date)


        # --- Build Query String Dynamically using REST API syntax ---
        query_parts = []
        if query.strip():
             # Wrap base query in parentheses if it contains spaces or operators, to be safe
             if ' ' in query.strip() or any(op in query for op in [' AND ', ' OR ', ' NOT ']):
                 query_parts.append(f"({query.strip()})")
             else:
                 query_parts.append(query.strip())


        # Add specific field filters
        if title and isinstance(title, str) and title.strip():
            query_parts.append(f'title:"{title.strip()}"')
        if item_type and isinstance(item_type, str) and item_type.strip():
             # Item types often need exact case and quotes
            query_parts.append(f'type:"{item_type.strip()}"')
        if owner and isinstance(owner, str) and owner.strip():
            query_parts.append(f'owner:"{owner.strip()}"')
        if snippet and isinstance(snippet, str) and snippet.strip():
            query_parts.append(f'snippet:"{snippet.strip()}"')
        if group_id and isinstance(group_id, str) and group_id.strip():
             query_parts.append(f'group:"{group_id.strip()}"')


        # Handle list-based filters
        tags_filter = _format_list_filter("tags", tags)
        if tags_filter: query_parts.append(tags_filter)

        typeKeywords_filter = _format_list_filter("typeKeywords", typeKeywords)
        if typeKeywords_filter: query_parts.append(typeKeywords_filter)

        categories_filter = _format_list_filter("categories", categories)
        if categories_filter: query_parts.append(categories_filter)


        # Date range handling (inclusive using milliseconds)
        def format_date_range_ms(field: str, start_ms: Optional[int], end_ms: Optional[int]) -> Optional[str]:
            if start_ms is None and end_ms is None:
                return None
            start = start_ms if start_ms is not None else "*"
            end = end_ms if end_ms is not None else "*"
            # Ensure start is not greater than end if both are specified
            if isinstance(start, int) and isinstance(end, int) and start > end:
                 raise ValueError(f"Start date cannot be after end date for {field}.")
            return f"{field}:[{start} TO {end}]"

        created_range = format_date_range_ms("created", created_start_ms, created_end_ms)
        if created_range: query_parts.append(created_range)


        # Handle Living Atlas focus: Apply ONLY if no specific owner/group/tags were provided
        specific_filters_provided = bool(owner or tags or group_id) # Removed orgid check
        if search_living_atlas_focused and not specific_filters_provided:
            # Add common LA owners/tags - adjust as needed based on current best practices
            la_filter = '(owner:esri OR owner:"esri_livingatlas" OR owner:"LivingAtlas" OR tags:"Living Atlas")'
            query_parts.append(la_filter)

        # Handle access filter based on search_outside_org
        # If searching outside org, explicitly add access:public for clarity and broader reach
        if search_outside_org:
            query_parts.append("access:public")
        # If search_outside_org is False, we rely on the API's default behavior
        # which searches within the user's org content when authenticated.

        # Combine all parts with AND
        final_query = " AND ".join(filter(None, query_parts)) # Filter out any None parts

        if not final_query:
             return "Error: No valid search criteria provided. Please specify a query or at least one filter."

        print(f"Executing ArcGIS Content search query: {final_query}")
        print(f"Searching outside organization: {search_outside_org}")

        # --- Execute Search ---
        # Pass the constructed query string to gis.content.search
        # The outside_org parameter controls whether to search beyond the user's org content
        search_results = gis.content.search(
            query=final_query,
            max_items=max_results,
            outside_org=search_outside_org
        )

        # --- Format Results ---
        if not search_results:
            # Provide more context in the "not found" message
            filters_used = [p for p in query_parts if p != query.strip()] # Show filters applied
            filters_str = f" with filters: [{', '.join(filters_used)}]" if filters_used else ""
            return f"No items found matching your criteria: '{query}'{filters_str}."

        output_results = []
        for item in search_results:
            # Safely get dates and format them back to YYYY-MM-DD HH:MM:S
            def format_timestamp(timestamp_ms):
                 if not timestamp_ms: return None
                 try:
                     # Convert milliseconds to seconds
                     timestamp_sec = timestamp_ms / 1000
                     return datetime.fromtimestamp(timestamp_sec).strftime('%Y-%m-%d %H:%M:%S')
                 except Exception:
                     return None # Handle potential errors during conversion

            created_str = format_timestamp(item.created)
            # modified_str = format_timestamp(item.modified) # Removed

            output_results.append({
                "title": item.title,
                "id": item.id,
                "type": item.type,
                "owner": item.owner,
                "snippet": item.snippet,
                "description": item.description, # Include description
                "tags": item.tags,
                "typeKeywords": getattr(item, 'typeKeywords', None), # Include typeKeywords if available
                "created": created_str,
                "access": item.access, # Include access level
                "url": item.url # Include item URL
            })

        return json.dumps(output_results, indent=2)

    except ValueError as ve: # Catch validation errors specifically
        return f"Input validation error: {str(ve)}"
    except Exception as e:
        # Log the full traceback for better debugging if possible in the environment
        # traceback.print_exc()
        return f"An unexpected error occurred during ArcGIS content search: {str(e)}"

In [ ]:
results = search_arcgis_content("", item_type="Feature Service", search_living_atlas_focused=True, max_results=10)
print(results)

Attempting to connect to https://www.arcgis.com as user mgi10015.23_bitmesragis...
Successfully connected to https://bitmesragis.maps.arcgis.com as user mgi10015.23_bitmesragis
Executing ArcGIS Content search query: type:"Feature Service" AND (owner:esri OR owner:"esri_livingatlas" OR owner:"LivingAtlas" OR tags:"Living Atlas") AND access:public
Searching outside organization: True
[
  {
    "title": "USA Soils Map Units",
    "id": "06e5fd61bdb6453fb16534c676e1c9b9",
    "type": "Feature Service",
    "owner": "esri",
    "snippet": "This feature layer displays soil map units of the 50 US States and its territories. It is derived from the US Department of Agriculture, Natural Resources Conservation Service SSURGO dataset.",
    "description": "Soil map units are the basic geographic unit of the <a href='https://www.nrcs.usda.gov/wps/portal/nrcs/detail/soils/survey/?cid=nrcs142p2_053627' target='_blank' rel='nofollow ugc noopener noreferrer'>Soil Survey Geographic Database</a> (SSURGO)

: 

In [ ]:
import os
import time
import zipfile
import pandas as pd
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
from arcgis.geometry import Geometry

import arcgis.features


def download_living_atlas_item_as_shapefile(item_id: str, save_path: str) -> str | None:
    """
    Attempts to download or export data from a Living Atlas item as a local Shapefile.

    Handles common data types like Shapefiles, Feature Services, Map Services,
    CSVs (with coordinates), and informs about types requiring manual conversion
    (FGDB, KML, LPKX) or types not convertible (Imagery, Tiles, Maps, Apps).

    Args:
        item_id: The Item ID of the Living Atlas content.
        gis: An authenticated arcgis.gis.GIS object connection.
        save_path: The local directory path where the Shapefile (or downloaded file)
                   should be saved.

    Returns:
        The full path to the created Shapefile if successful, otherwise None.
        For types like FGDB, KML, LPKX, it downloads the original file but returns None
        as a Shapefile wasn't directly created.
    """
    gis = get_gis_connection()
    if not gis:
        return "Error: Failed to establish connection to ArcGIS. Check API Key/Credentials."

    try:
        print(f"--- Processing Item ID: {item_id} ---")
        item = gis.content.get(item_id)
        if not item:
            print(f"Error: Item with ID '{item_id}' not found.")
            return None

        item_type = item.type
        item_title_cleaned = "".join(c if c.isalnum() or c in (' ', '_') else '_' for c in item.title).rstrip()
        print(f"Found Item: '{item.title}' (Type: {item_type})")

        # Ensure save directory exists
        os.makedirs(save_path, exist_ok=True)
        
        # --- Handle specific item types ---

        # 1. Direct Shapefile Download
        if item_type == 'Shapefile':
            print("Item is a Shapefile. Downloading...")
            try:
                downloaded_zip_path = item.download(save_path=save_path)
                if not downloaded_zip_path:
                     print("Error: Download failed.")
                     return None
                     
                print(f"Downloaded zip: {downloaded_zip_path}")
                # Unzip
                shapefile_name = None
                extract_folder = os.path.join(save_path, item_title_cleaned + "_shp")
                os.makedirs(extract_folder, exist_ok=True)
                with zipfile.ZipFile(downloaded_zip_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_folder)
                    # Find the .shp file
                    for file in zip_ref.namelist():
                        if file.lower().endswith('.shp'):
                            shapefile_name = os.path.join(extract_folder, file)
                            break
                os.remove(downloaded_zip_path) # Clean up zip file
                if shapefile_name:
                    print(f"Successfully downloaded and unzipped Shapefile to: {shapefile_name}")
                    return shapefile_name
                else:
                    print("Error: Could not find .shp file within the downloaded zip.")
                    return None
            except Exception as e:
                print(f"Error downloading/unzipping Shapefile: {e}")
                return None

        # 2. Feature Service / Map Service / Feature Collection - Try Export first
        elif item_type in ['Feature Service', 'Map Service', 'Feature Collection']:
            print("Item is a Service/Collection. Attempting export to Shapefile...")
            export_succeeded = False
            exported_shp_path = None
            try:
                # Check if export is possible (basic check, might still fail)
                if not item.access or not 'shared' in item.access.lower():
                     if not (gis.users.me and item.owner == gis.users.me.username):
                         print("Warning: Item might not be public or exportable.")

                # Try exporting the item
                result_item = item.export(title=f"Export_{item_title_cleaned[:50]}_{int(time.time())}",
                                          export_format="Shapefile",
                                          wait=True,
                                          enforce_job_ownership=False) # Try running as data owner if needed
                
                if result_item:
                    print(f"Export successful (Created Item ID: {result_item.itemid}). Downloading exported Shapefile...")
                    downloaded_zip_path = result_item.download(save_path=save_path)
                    result_item.delete() # Clean up the exported item on AGOL/Portal

                    if not downloaded_zip_path:
                         print("Error: Download of exported file failed.")
                         return None # Exit here, don't try query

                    # Unzip
                    shapefile_name = None
                    extract_folder = os.path.join(save_path, item_title_cleaned + "_exported_shp")
                    os.makedirs(extract_folder, exist_ok=True)
                    with zipfile.ZipFile(downloaded_zip_path, 'r') as zip_ref:
                        zip_ref.extractall(extract_folder)
                        for file in zip_ref.namelist():
                            if file.lower().endswith('.shp'):
                                exported_shp_path = os.path.join(extract_folder, file)
                                break
                    os.remove(downloaded_zip_path) # Clean up zip
                    
                    if exported_shp_path:
                         print(f"Successfully exported and downloaded Shapefile to: {exported_shp_path}")
                         return exported_shp_path
                    else:
                         print("Error: Could not find .shp file within the exported zip.")
                         return None # Export worked, but unzip/find failed
                else:
                    print("Export process did not return a result item.")
                    # Proceed to try querying below

            except Exception as export_err:
                print(f"Warning: Export to Shapefile failed. Reason: {export_err}")
                # Fall through to try querying if export failed

            # 2b. Fallback: Query first Feature Layer if export failed or wasn't possible
            print("Attempting to query the first Feature Layer...")
            if item.layers:
                try:
                    # Find the first actual *Feature* Layer
                    first_feature_layer = None
                    for lyr in item.layers:
                        # Check if it behaves like a feature layer (has query method)
                        if hasattr(lyr, 'query') and isinstance(lyr, FeatureLayer):
                             first_feature_layer = lyr
                             break 
                             
                    if not first_feature_layer:
                        print("No suitable Feature Layer found within the item to query.")
                        return None

                    print(f"Querying layer: '{first_feature_layer.properties.name}'")
                    # Query all features (potentially large!) - consider adding limits/filters
                    sdf = first_feature_layer.query(where='1=1', return_geometry=True).sdf
                    
                    if sdf.empty:
                        print("Warning: Query returned no features from the first layer.")
                        return None
                        
                    # Save SDF to Shapefile
                    layer_name_cleaned = "".join(c if c.isalnum() or c in (' ', '_') else '_' for c in first_feature_layer.properties.name).rstrip()
                    output_filename_base = os.path.join(save_path, f"{item_title_cleaned}_{layer_name_cleaned}")
                    output_shp_path = f"{output_filename_base}.shp"
                    
                    print(f"Saving queried features to: {output_shp_path}")
                    sdf.spatial.to_featureclass(location=output_shp_path)
                    
                    print(f"Successfully queried first layer and saved to Shapefile: {output_shp_path}")
                    return output_shp_path

                except Exception as query_err:
                    print(f"Error querying/saving first layer: {query_err}")
                    return None
            else:
                print("No layers found in the item to query.")
                return None

        # 3. CSV - Try Download and Convert
        elif item_type == 'CSV':
            print("Item is a CSV. Downloading and attempting conversion...")
            try:
                csv_path = item.download(save_path=save_path)
                print(f"Downloaded CSV to: {csv_path}")
                
                # Attempt conversion using pandas and arcgis.features
                df = pd.read_csv(csv_path)
                
                # Guess coordinate columns (case-insensitive)
                lat_col, lon_col = None, None
                x_col, y_col = None, None
                possible_lats = ['latitude', 'lat', 'y', 'ycenter']
                possible_lons = ['longitude', 'lon', 'long', 'x', 'xcenter']
                
                for col in df.columns:
                    col_lower = col.lower()
                    if col_lower in possible_lats: lat_col = col
                    if col_lower in possible_lons: lon_col = col
                
                # Prefer lat/lon if found
                if lat_col and lon_col:
                    x_col, y_col = lon_col, lat_col # SDF uses x, y order
                    print(f"Found potential coordinate columns: X='{x_col}', Y='{y_col}'")
                    # Check if data looks numeric
                    if pd.api.types.is_numeric_dtype(df[x_col]) and pd.api.types.is_numeric_dtype(df[y_col]):
                         # Drop rows with invalid coordinates before conversion
                         df.dropna(subset=[x_col, y_col], inplace=True)
                         # Convert to Spatially Enabled DataFrame (assuming WGS84 for lat/lon)
                         sdf = pd.DataFrame.spatial.from_xy(df, x_col, y_col, sr=4326)
                         
                         # Save to Shapefile
                         output_filename_base = os.path.join(save_path, item_title_cleaned + "_csv_converted")
                         output_shp_path = f"{output_filename_base}.shp"
                         sdf.spatial.to_featureclass(location=output_shp_path)
                         print(f"Successfully converted CSV and saved to Shapefile: {output_shp_path}")
                         os.remove(csv_path) # Clean up original CSV
                         return output_shp_path
                    else:
                         print("Coordinate columns found but are not numeric. Cannot convert.")
                else:
                    print("Could not automatically detect suitable coordinate columns (lat/lon or x/y).")

                print("CSV downloaded, but automatic conversion to Shapefile failed or wasn't possible.")
                return None # Return None as shapefile goal wasn't met

            except Exception as e:
                print(f"Error downloading or converting CSV: {e}")
                return None

        # 4. Types Requiring Manual Conversion after Download
        elif item_type in ['File Geodatabase', 'KML', 'Layer Package', 'Scene Layer Package']:
             print(f"Item type is '{item_type}'. Downloading original file...")
             try:
                 downloaded_path = item.download(save_path=save_path)
                 print(f"Downloaded original file to: {downloaded_path}")
                 print("Manual conversion required: Use ArcGIS Pro or other GIS software (like QGIS, or libraries like geopandas/fiona) to extract/convert data from this file to Shapefile format.")
                 return None # Return None as shapefile wasn't directly created
             except Exception as e:
                 print(f"Error downloading {item_type}: {e}")
                 return None

        # 5. Types Not Convertible to Shapefile
        elif item_type in ['Vector Tile Service', 'Image Service', 'Raster Layer', 'Map Image Layer', 'Tile Layer', 'Scene Service']:
            print(f"Item type '{item_type}' is not vector feature data and cannot be directly converted to a Shapefile by this function.")
            return None

        # 6. Complex Container Types
        elif item_type in ['Web Map', 'Web Scene']:
            print(f"Item type '{item_type}' is a map/scene configuration. It contains layers but isn't directly downloadable as a single Shapefile.")
            print("You may need to inspect the map/scene's layers (e.g., via item.get_data()) and run this function on the individual Feature Layer item IDs found within.")
            return None
            
        # 7. Other Unsupported Types
        else:
            print(f"Item type '{item_type}' is not supported for direct Shapefile download/export by this function (e.g., Apps, Dashboards, Stories, Documents).")
            return None

    except Exception as e:
        print(f"An unexpected error occurred processing item {item_id}: {e}")
        import traceback
        traceback.print_exc() # Print detailed traceback for debugging
        return None

In [25]:
download_living_atlas_item_as_shapefile(item_id="77f61d02dedf4e858ab8af36e7cdd35a", save_path="./downloaded_data")

Attempting to connect to https://www.arcgis.com as user mgi10015.23_bitmesragis...
Successfully connected to https://bitmesragis.maps.arcgis.com as user mgi10015.23_bitmesragis
--- Processing Item ID: 77f61d02dedf4e858ab8af36e7cdd35a ---
An unexpected error occurred processing item 77f61d02dedf4e858ab8af36e7cdd35a: Subscription is canceled, the item is not accessible
(Error Code: 403)


Traceback (most recent call last):
  File "C:\Users\raiha\AppData\Local\Temp\ipykernel_13808\2097311314.py", line 36, in download_living_atlas_item_as_shapefile
    item = gis.content.get(item_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\masters_project\ArcGIS_AI\envs\arcgis_llm\Lib\site-packages\arcgis\gis\__init__.py", line 7350, in get
    raise e
  File "d:\masters_project\ArcGIS_AI\envs\arcgis_llm\Lib\site-packages\arcgis\gis\__init__.py", line 7340, in get
    item = self._portal.get_item(itemid)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\masters_project\ArcGIS_AI\envs\arcgis_llm\Lib\site-packages\arcgis\gis\_impl\_portalpy.py", line 1438, in get_item
    return self.con.post("content/items/" + itemid, self._postdata())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\masters_project\ArcGIS_AI\envs\arcgis_llm\Lib\site-packages\arcgis\gis\_impl\_con\_connection.py", line 1504, in post
    return self._handle_response(
           ^^^^^

In [13]:
get_living_atlas_item_data(item_id="3490f1af998d44f992a3c55cf41270b5", save_path="./downloaded_data")

Attempting to connect to https://www.arcgis.com as user mgi10015.23_bitmesragis...
Successfully connected to https://bitmesragis.maps.arcgis.com as user mgi10015.23_bitmesragis
Attempting to access Living Atlas item with ID: 3490f1af998d44f992a3c55cf41270b5
Found item: 'OS Open USRN (Download Only)' (Type: Feature Service, Owner: EsriUKContent, Access: public)
Attempting to download item 'OS Open USRN (Download Only)' to ./downloaded_data...


"Successfully downloaded item 3490f1af998d44f992a3c55cf41270b5 ('OS Open USRN (Download Only)') to: ./downloaded_data\\OS_Open_USRN_Download"